# Question 1

In [28]:
import numpy as np
import matplotlib.pyplot as plt
import galsim
import galsim.hsm
from pathlib import Path
 
 
# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA_DIR = Path("data") # Directory where input data is stored
OUTPUT_DIR = Path("outputs") # Directory where outputs will be saved
OUTPUT_DIR.mkdir(exist_ok=True) # Create output directory if it doesn't exist
 
STAMPS_FILE = DATA_DIR / "galaxy_stamps.fits" # FITS file containing galaxy stamps

## Question 1a

Ellipticity is defined in terms of quadrupole moments by

\begin{equation}
    \epsilon_1=\frac{Q_{11}-Q_{22}}{Q_{11}+Q_{22}+2\sqrt{Q_{11}Q_{22}-Q_{12}^2}},
\end{equation}

\begin{equation}
    \epsilon_2=\frac{2 Q_{12}}{Q_{11}+Q_{22}+2\sqrt{Q_{11}Q_{22}-Q_{12}^2}},
\end{equation}

where

\begin{equation}
    Q_{pq}=\frac{\int dxdyI(x,y)(p-\bar{p})(q-\bar{q})}{\int dxdyI(x,y)},
\end{equation}

and

\begin{equation}
    \bar{x}=\frac{\int dxdy I(x,y) x}{\int dxdy I(x,y)}.
\end{equation}

These equations can be modified to include the effects of weighting the pixels. The expressions for ellipticity are unchanged, but the quadrupole moments become

\begin{equation}
    Q^W_{ij}=\frac{\int dxdy W(x,y)I(x,y)(p-\bar{p})(q-\bar{q})}{\int dxdy W(x,y)I(x,y)},
\end{equation}

and

\begin{equation}
    \bar{p}=\frac{\int dxdy W(x,y) I(x,y) p}{\int dxdy W(x,y) I(x,y)}.
\end{equation}

For the weight function, we use a circular Gaussian:

\begin{equation}
    W(x,y)=\exp(-\frac{1}{2\sigma^2}((x-\bar{x})^2+(y-\bar{y})^2)),
\end{equation}

where the standard deviation is related to the FWHM by

\begin{equation}
    \text{FWHM}=2\sqrt{2\ln2}\sigma.
\end{equation}

In [29]:
def _pixel_grids(image: galsim.Image):
    """
    Return (x, y) pixel-offset arrays relative to the image centre.
    Parameters
    ----------
    image : galsim.Image
        The image for which to compute the pixel grids.
    Returns
    -------
    x, y : np.ndarray
        2D arrays of shape (ny, nx) containing the x and y pixel offsets
        from the image centre.
    """

    arr = image.array # Get the image array
    ny, nx = arr.shape # Get the dimensions of the image
    cx, cy = (nx - 1) / 2.0, (ny - 1) / 2.0 # Compute the centre coordinates
    x_idx  = np.arange(nx, dtype=float) - cx # Create x pixel offsets
    y_idx  = np.arange(ny, dtype=float) - cy # Create y pixel offsets
    x, y   = np.meshgrid(x_idx, y_idx)   # Shape (ny, nx)
    return x, y

In [30]:
def unweighted_ellipticity(image: galsim.Image):
    """
    Estimate (epsilon_1, epsilon_2) from unweighted second-order moments.
 
        Q_ij = sum_{pq} I(p,q) * dx_i * dx_j
        epsilon_1 = (Q11 - Q22) / (Q11 + Q22 + 2 * sqrt(Q11 * Q22 - Q12^2))
        epsilon_2 = 2 * Q12 / (Q11 + Q22 + 2 * sqrt(Q11 * Q22 - Q12^2))
 
    Measurements with |epsilon| > 1 (noise-dominated / diverging) are returned
    as (nan, nan) because they indicate a failed centroid or a stamp where
    the noise dominates the moment denominator.
 
    Returns
    -------
    (epsilon_1, epsilon_2) : floats, or (nan, nan) on failure.
    """
    arr   = image.array.astype(float) # Get the image array as float for calculations
    x, y  = _pixel_grids(image) # Get the pixel-offset relative to centre grids
 
    I_sum = arr.sum() # Total intensity (zeroth moment)
    if I_sum <= 0:
        return np.nan, np.nan # No signal, so return NaN for ellipticity
 
    # Centroid computations (first moments)
    xbar = (arr * x).sum() / I_sum # Centroid x-coordinate
    ybar = (arr * y).sum() / I_sum # Centroid y-coordinate
    dx   = x - xbar # Pixel offsets from centroid in x
    dy   = y - ybar # Pixel offsets from centroid in y
 
    # Second moments
    Q11   = (arr * dx * dx).sum() / I_sum # Q11 = sum(I * dx^2)
    Q22   = (arr * dy * dy).sum() / I_sum # Q22 = sum(I * dy^2)
    Q12   = (arr * dx * dy).sum() / I_sum # Q12 = sum(I * dx * dy)
    det  = Q11 * Q22 - Q12 ** 2 # Determinant of the second moment matrix

    if det < 0:
        return np.nan, np.nan
    
    denom = Q11 + Q22 + 2.0 * np.sqrt(det) # Denominator for ellipticity calculation
 
    if denom <= 0:
        return np.nan, np.nan # Denominator is zero or negative, likely noise-dominated, so return NaN
 
    epsilon_1 = (Q11 - Q22) / denom # Ellipticity component epsilon_1
    epsilon_2 = 2 * Q12 / denom # Ellipticity component epsilon_2
 
    # Physical bound: |epsilon| <= 1 for any non-negative intensity profile.
    # Values outside this range signal a diverging / noise-dominated stamp.
    if abs(epsilon_1) > 1.0 or abs(epsilon_2) > 1.0:
        return np.nan, np.nan # Return NaN for unphysical ellipticity values
 
    return epsilon_1, epsilon_2

In [31]:
def _gaussian_weight(dx, dy, sigma):
    """
    Circular Gaussian W = exp(-(dx^2 + dy^2) / (2 sigma^2)).
    
    Parameters
    ----------
    dx, dy : np.ndarray
        Pixel offsets from the centroid in x and y directions.
    sigma : float
        Standard deviation of the Gaussian weight function in pixels.

    Returns
    -------
    W : np.ndarray
        2D array of the same shape as dx and dy containing the Gaussian weights.
    """

    return np.exp(-(dx**2 + dy**2) / (2.0 * sigma**2))
 
 
def weighted_ellipticity_raw(image: galsim.Image, fwhm_px: float = 4.0, max_iter: int = 20, tol: float = 1e-3):
    """
    Estimate (epsilon_1, epsilon_2) from weighted second-order moments,
    using the same lecture convention as the unweighted estimator but with
    intensity I(x,y) replaced by W(x,y)*I(x,y):

        Q_pq^W = [sum_{xy} W(x,y) * I(x,y) * (p - pbar^W) * (q - qbar^W)]
                 / [sum_{xy} W(x,y) * I(x,y)]

        epsilon_1 = (Q_xx^W - Q_yy^W) / D^W
        epsilon_2 = 2 * Q_xy^W         / D^W
        D^W = Q_xx^W + Q_yy^W + 2 * sqrt(Q_xx^W * Q_yy^W - (Q_xy^W)^2)

    The weight function is a circular Gaussian:
        W(x, y) = exp(-((x - xbar^W)^2 + (y - ybar^W)^2) / (2*sigma^2))
    with sigma = FWHM / (2*sqrt(2*ln2)).

    The weighted centroid (xbar^W, ybar^W) is found iteratively:
        1. Initialise with the unweighted centroid.
        2. Centre the Gaussian weight on the current centroid estimate.
        3. Recompute the centroid from weighted first moments:
               xbar^W = sum(W * I * x) / sum(W * I)
        4. Repeat until the centroid shifts by less than tol pixels.

    This ensures the weight function is self-consistently centred on the
    weighted centroid, as required by the definition of Q_pq^W.

    Parameters
    ----------
    image    : galsim.Image
    fwhm_px  : FWHM of the Gaussian weight function in pixels (default 4.0)
    max_iter : maximum number of centroid iterations (default 20)
    tol      : convergence threshold in pixels (default 1e-3)

    Returns
    -------
    (epsilon_1, epsilon_2) : floats, or (nan, nan) on failure.
    """
    
    arr  = image.array.astype(float) # Get the image array as float for calculations
    x, y = _pixel_grids(image) # Get the pixel-offset relative to centre grids
 
    sigma = fwhm_px / (2.0 * np.sqrt(2.0 * np.log(2.0))) # Convert FWHM to sigma for Gaussian weight
 
    # --- Step 1: initialise centroid from unweighted first moments ---
    I_sum = arr.sum()
    if I_sum <= 0:
        return np.nan, np.nan
    xbar = (arr * x).sum() / I_sum
    ybar = (arr * y).sum() / I_sum
 
    # --- Step 2: iterate to find the weighted centroid ---
    for _ in range(max_iter):
        dx = x - xbar
        dy = y - ybar
 
        W      = _gaussian_weight(dx, dy, sigma)
        WI     = W * arr
        WI_sum = WI.sum()
        if WI_sum <= 0:
            return np.nan, np.nan
 
        # Weighted first moments give the new centroid estimate
        xbar_new = (WI * x).sum() / WI_sum
        ybar_new = (WI * y).sum() / WI_sum
 
        # Check convergence: centroid shift in pixels
        shift = np.sqrt((xbar_new - xbar)**2 + (ybar_new - ybar)**2)
        xbar, ybar = xbar_new, ybar_new
 
        if shift < tol:
            break
    
    dx   = x - xbar # Pixel offsets from the centroid in x direction
    dy   = y - ybar # Pixel offsets from the centroid in y direction
 
    # Gaussian weight centred on the unweighted centroid
    W = _gaussian_weight(dx, dy, sigma) # Compute the Gaussian weights for each pixel based on the offsets and sigma
    WI = W * arr # Weighted intensity for each pixel
    WI_sum = WI.sum() # Total weighted intensity (zeroth moment with weights)
    if WI_sum <= 0:
        return np.nan, np.nan # No signal in the weighted image, so return NaN for ellipticity
 
    # Weighted second moments
    Q11   = (WI * dx * dx).sum() / WI_sum # Q11 = sum(W * I * dx^2)
    Q22   = (WI * dy * dy).sum() / WI_sum # Q22 = sum(W * I * dy^2)
    Q12   = (WI * dx * dy).sum() / WI_sum # Q12 = sum(W * I * dx * dy)
    det  = Q11 * Q22 - Q12 ** 2 # Determinant of the second moment matrix

    if det < 0:
        return np.nan, np.nan # Determinant is negative, likely noise-dominated, so return NaN for ellipticity
    
    denom = Q11 + Q22 + 2.0 * np.sqrt(det) # Denominator for ellipticity calculation
 
    if denom <= 0:
        return np.nan, np.nan # Denominator is zero or negative, likely noise-dominated, so return NaN
 
    epsilon_1 = (Q11 - Q22) / denom # Ellipticity component epsilon_1
    epsilon_2 = 2 * Q12 / denom # Ellipticity component epsilon_2

    # Physical bound: |epsilon| <= 1 for any non-negative intensity profile.
    # Values outside this range signal a diverging / noise-dominated stamp.
    if abs(epsilon_1) > 1.0 or abs(epsilon_2) > 1.0:
        return np.nan, np.nan # Return NaN for unphysical ellipticity values
    
    return epsilon_1, epsilon_2

In [32]:
def hsm_ellipticity(image: galsim.Image):
    """
    Estimate (g1, g2, sigma_g) and the adaptive moment size sigma_g using GalSim's
    HSM algorithm (Hirata & Seljak 2003; Mandelbaum et al. 2005).

    HSM iteratively adapts an elliptical Gaussian weight to match the
    galaxy shape.  The returned size moments_sigma is the best-fit
    circular Gaussian sigma in pixels, which is used for the analytical
    response correction of the fixed-weight estimators.

    Returns
    -------
    (g1, g2, sigma_g) : floats, or (nan, nan, nan) if HSM fails.
    """
    try:
        result = galsim.hsm.FindAdaptiveMom(image)
        return (result.observed_shape.g1,
                result.observed_shape.g2,
                result.moments_sigma)
    except galsim.GalSimHSMError:
        return np.nan, np.nan, np.nan

In [33]:
def calibrate_weighted_response(
        fwhm_px: float = 4.0,
        stamp_size: int = 48,
        pixel_scale: float = 1.0,
        shear_cal: float = 0.05,
        n_per_hlr: int = 1000,
        seed: int = 42,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Build a lookup table of weighted-moment shear response R(sigma_g) by
    simulating noiseless Exponential-profile galaxies with a known applied
    shear and measuring the ratio of recovered to input ellipticity.

    For each half-light radius in a pre-defined grid, n_per_hlr noiseless
    galaxy images are drawn with GalSim (Exponential profile, shear g1=shear_cal,
    g2=0), passed through the same weighted-moment pipeline used on the data,
    and the response is estimated as

        R = <epsilon_1_measured> / shear_cal

    The interpolation axis is the median HSM sigma_g for each hlr bin, so
    that the calibration can be applied per galaxy using its HSM size.

    Parameters
    ----------
    fwhm_px    : FWHM of the Gaussian weight function in pixels (must match
                 the value used in weighted_ellipticity_raw).
    stamp_size : stamp side length in pixels (must match data stamps).
    pixel_scale: arcsec per pixel (must match data).
    shear_cal  : applied reduced shear magnitude; large enough for S/N but
                 small enough to stay in the weak-lensing regime (g << 1).
    n_per_hlr  : number of noiseless realisations per grid point.
    seed       : RNG seed for reproducibility.

    Returns
    -------
    sigma_g_nodes : (N_grid,) HSM sigma_g values [pixels] — interpolation x-axis.
    R_nodes       : (N_grid,) response values        — interpolation y-axis.
    Both arrays are sorted by sigma_g_nodes and have NaN entries removed.
    """
    sigma_w = fwhm_px / (2.0 * np.sqrt(2.0 * np.log(2.0)))

    # Grid of half-light radii spanning sub-pixel to well-resolved galaxies.
    # The upper end (hlr=10px) ensures coverage for the largest galaxies in
    # the data; the lower end probes the pixel-dominated regime.
    hlr_grid = np.array([0.4, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 7.0, 10.0])

    sigma_g_nodes = []
    R_nodes = []

    print(f"Calibrating weighted-moment response on {len(hlr_grid)} galaxy sizes...")
    for hlr in hlr_grid:
        e1_list = []
        sg_list = []
        for _ in range(n_per_hlr):
            # Noiseless Exponential galaxy with known shear applied
            gal = galsim.Exponential(half_light_radius=hlr * pixel_scale)
            gal = gal.shear(g1=shear_cal, g2=0.0)
            im = gal.drawImage(
                scale=pixel_scale, nx=stamp_size, ny=stamp_size, method="auto"
            )
            # Weighted moments (same pipeline as on the data)
            e1, _ = weighted_ellipticity_raw(im, fwhm_px=fwhm_px)
            e1_list.append(e1)
            # HSM size used as the interpolation key
            try:
                sg_list.append(galsim.hsm.FindAdaptiveMom(im).moments_sigma)
            except galsim.GalSimHSMError:
                sg_list.append(np.nan)

        e1_arr = np.array(e1_list)
        valid = np.isfinite(e1_arr)
        if valid.sum() < 10:
            continue  # Too few valid measurements at this size; skip

        R = e1_arr[valid].mean() / shear_cal
        sg_med = np.nanmedian(sg_list)

        if not np.isfinite(sg_med) or not np.isfinite(R) or R <= 0:
            continue  # Skip size bins where HSM itself fails

        sigma_g_nodes.append(sg_med)
        R_nodes.append(R)
        print(f"  hlr={hlr:.2f}px  sigma_g={sg_med:.3f}px  R={R:.4f}")

    sigma_g_nodes = np.array(sigma_g_nodes)
    R_nodes = np.array(R_nodes)

    # Ensure the interpolation axis is strictly increasing (required by np.interp)
    order = np.argsort(sigma_g_nodes)
    return sigma_g_nodes[order], R_nodes[order]


def apply_simulation_calibration(
        epsilon: np.ndarray,
        sigma_gs: np.ndarray,
        sigma_g_nodes: np.ndarray,
        R_nodes: np.ndarray,
) -> np.ndarray:
    """
    Divide each galaxy's raw weighted ellipticity by its per-galaxy response
    R_i, interpolated from the simulation calibration table.

    np.interp is used for linear interpolation with boundary clamping:
    galaxies outside the calibrated sigma_g range receive the response of
    the nearest calibration node rather than an extrapolated (potentially
    unphysical) value.

    Parameters
    ----------
    epsilon        : (N, 2) raw ellipticity array from weighted_ellipticity_raw.
    sigma_gs       : (N,) per-galaxy HSM sigma_g values [pixels].
    sigma_g_nodes  : (M,) calibration x-axis (sorted).
    R_nodes        : (M,) calibration y-axis.

    Returns
    -------
    epsilon_corr : (N, 2) calibrated ellipticity array; NaN where sigma_g
                   or the raw ellipticity is NaN.
    """
    # Replace NaN sigma_gs with the median before interpolating,
    # so weighted moments are not discarded solely due to HSM failure
    sigma_gs_filled = sigma_gs.copy()
    nan_sg = ~np.isfinite(sigma_gs_filled)
    sigma_gs_filled[nan_sg] = np.nanmedian(sigma_gs)

    R_per_galaxy = np.interp(sigma_gs_filled, sigma_g_nodes, R_nodes)

    corr = epsilon.copy()
    valid = np.isfinite(epsilon[:, 0]) & np.isfinite(R_per_galaxy) & (R_per_galaxy > 0)
    corr[valid, 0] /= R_per_galaxy[valid]
    corr[valid, 1] /= R_per_galaxy[valid]
    corr[~valid] = np.nan
    return corr

In [34]:
def measure_all_ellipticities(stamps_path: Path):
    """
    Load all galaxy stamps and compute (e1, e2) for each estimator.

    For the weighted-moment estimator the raw ellipticities are corrected
    for the multiplicative shear response bias introduced by the fixed
    circular Gaussian weight function. The response R(sigma_g) is measured
    from GalSim simulations via calibrate_weighted_response and applied
    per-galaxy via apply_simulation_calibration.

    The unweighted estimator has unit response in the noiseless limit and
    is returned uncorrected. HSM returns adaptive-moment ellipticities that
    are already self-calibrated by construction.

    Returns
    -------
    dict with keys "unweighted", "weighted", "hsm", each an ndarray
    of shape (N, 2) containing (e1, e2) per galaxy (NaN on failure).
    """
    galaxy_ims = galsim.fits.readMulti(str(stamps_path))
    n_gals     = len(galaxy_ims)
    print(f"Loaded {n_gals} galaxy stamps from {stamps_path}")

    e_unw    = np.full((n_gals, 2), np.nan)
    e_wgt    = np.full((n_gals, 2), np.nan)
    e_hsm    = np.full((n_gals, 2), np.nan)
    sigma_gs = np.full(n_gals,      np.nan)  # per-galaxy HSM size [pixels]

    for i, im in enumerate(galaxy_ims):
        e_unw[i]       = unweighted_ellipticity(im)
        e_wgt[i]       = weighted_ellipticity_raw(im, fwhm_px=4.0)
        e1, e2, sg     = hsm_ellipticity(im)
        e_hsm[i]       = e1, e2
        sigma_gs[i]    = sg

    print(f"\nMedian HSM galaxy size: sigma_g = {np.nanmedian(sigma_gs):.3f} px")

    # --- Simulation-based response calibration for the weighted estimator ---
    # The weight FWHM must match the value used in weighted_ellipticity_raw.
    sigma_g_nodes, R_nodes = calibrate_weighted_response(fwhm_px=4.0)

    print(f"\nCalibration range: sigma_g in [{sigma_g_nodes.min():.2f}, "
          f"{sigma_g_nodes.max():.2f}] px")
    print(f"Calibration R range: [{R_nodes.min():.3f}, {R_nodes.max():.3f}]")

    e_wgt_cal = apply_simulation_calibration(e_wgt, sigma_gs, sigma_g_nodes, R_nodes)

    results = {
        "unweighted": e_unw,
        "weighted":   e_wgt_cal,
        "hsm":        e_hsm,
    }

    # Summary statistics
    print("\nSummary statistics:")
    for name, arr in results.items():
        valid   = np.isfinite(arr[:, 0])
        n_valid = valid.sum()
        print(
            f"  {name:12s}: {n_valid:6d}/{n_gals} valid | "
            f"<e1> = {arr[valid, 0].mean():+.4f}  σ(e1) = {arr[valid, 0].std():.4f} | "
            f"<e2> = {arr[valid, 1].mean():+.4f}  σ(e2) = {arr[valid, 1].std():.4f}"
        )

    return results

In [35]:
def plot_ellipticity_comparison(results: dict, output_dir: Path):
    """
    Three diagnostic figures:
      1. Scatter e1 vs e2 per method (physical ±1 window).
      2. Histograms of e1 and e2 per method.
      3. Direct scatter of each method vs HSM, with 1:1 line and Pearson r.
    """
    methods = ["unweighted", "weighted", "hsm"]
    colours = {"unweighted": "steelblue", "weighted": "tomato", "hsm": "seagreen"}
    labels  = {
        "unweighted": "Unweighted moments",
        "weighted": "Weighted moments (Gaussian, FWHM=4px)",
        "hsm": "HSM (Hirata-Seljak-Mandelbaum)",
    }
 
    # ---- 1. Scatter e1 vs e2 (physical window) ------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True, sharex=True)
    for ax, method in zip(axes, methods):
        arr = results[method]
        valid = np.isfinite(arr[:, 0])
        ax.scatter(arr[valid, 0], arr[valid, 1], s=3, alpha=0.35, color=colours[method], rasterized=True)
        ax.axhline(0, lw=0.7, color="k", ls="--")
        ax.axvline(0, lw=0.7, color="k", ls="--")
        ax.set_xlim(-1, 1)
        ax.set_ylim(-1, 1)
        ax.set_xlabel(r"$\epsilon_1$", fontsize=13)
        ax.set_title(labels[method], fontsize=10)
        n_valid = valid.sum()
        ax.text(0.97, 0.97, f"N={n_valid}/{len(valid)}", transform=ax.transAxes, ha="right", va="top", fontsize=9)
    axes[0].set_ylabel(r"$\epsilon_2$", fontsize=13)
    fig.suptitle(r"Ellipticity estimates per method: $\epsilon_1$ vs $\epsilon_2$", fontsize=13)
    fig.tight_layout()
    out = output_dir / "q1_scatter_e1_e2.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out}")
 
    # ---- 2. Histograms -------------------------------------------------------
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
    bins = np.linspace(-1, 1, 60)
    for col, method in enumerate(methods):
        arr = results[method]
        valid = np.isfinite(arr[:, 0])
        for row, comp in enumerate(["\epsilon_1", "\epsilon_2"]):
            ax = axes[row, col]
            data = arr[valid, row]
            # Clip to [-1,1] for histogram (unweighted can have corrected
            # values slightly outside if R<1 but these are valid)
            data_clipped = np.clip(data, -1, 1)
            ax.hist(data_clipped, bins=bins, color=colours[method], alpha=0.7, density=True)
            ax.axvline(data.mean(), color="k", lw=1.5, ls="--", label=f"mean = {data.mean():.4f}")
            ax.set_title(f"{labels[method]}\n" f"${comp}$   σ = {data.std():.4f}", fontsize=9,
            )
            ax.legend(fontsize=8)
            if row == 1:
                ax.set_xlabel(f"${comp}$", fontsize=12)
            if col == 0:
                ax.set_ylabel("Density", fontsize=11)
    fig.suptitle("Ellipticity distributions per method", fontsize=13)
    fig.tight_layout()
    out = output_dir / "q1_histograms.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out}")
 
    # ---- 3. Method vs HSM scatter -------------------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(11, 10))
    e_hsm = results["hsm"]
    hsm_valid = np.isfinite(e_hsm[:, 0])
 
    compare_pairs = [
        ("unweighted", 0, "\epsilon_1"),
        ("unweighted", 1, "\epsilon_2"),
        ("weighted",   0, "\epsilon_1"),
        ("weighted",   1, "\epsilon_2"),
    ]
    for ax, (method, idx, comp_label) in zip(axes.flat, compare_pairs):
        arr = results[method]
        both_valid = hsm_valid & np.isfinite(arr[:, 0])
        x_vals = e_hsm[both_valid, idx]
        y_vals = arr[both_valid, idx]
 
        ax.scatter(x_vals, y_vals, s=3, alpha=0.35, color=colours[method], rasterized=True)
        lim = 0.5
        ax.plot([-lim, lim], [-lim, lim], "k--", lw=0.9, label="1:1")
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_xlabel(f"HSM  ${comp_label}$", fontsize=12)
        ax.set_ylabel(f"{labels[method].split('(')[0].strip()}  ${comp_label}$", fontsize=11)
        ax.legend(fontsize=8)
 
        r = np.corrcoef(x_vals, y_vals)[0, 1]
        ax.text(0.04, 0.95, f"r = {r:.3f}", transform=ax.transAxes, fontsize=9, va="top")
 
    fig.suptitle("Method comparison vs HSM", fontsize=13)
    fig.tight_layout()
    out = output_dir / "q1_comparison_vs_hsm.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {out}")

In [36]:
def main():
    results = measure_all_ellipticities(STAMPS_FILE) # Measure ellipticities for all galaxy stamps using the defined estimators
    plot_ellipticity_comparison(results, OUTPUT_DIR) # Generate diagnostic plots comparing the ellipticity estimates from different methods
 
    # Save for use in Q2
    np.save(OUTPUT_DIR / "ellipticities_unweighted.npy", results["unweighted"]) # Save unweighted ellipticity results to a .npy file in the outputs directory
    np.save(OUTPUT_DIR / "ellipticities_weighted.npy", results["weighted"]) # Save weighted ellipticity results to a .npy file in the outputs directory
    np.save(OUTPUT_DIR / "ellipticities_hsm.npy", results["hsm"]) # Save HSM ellipticity results to a .npy file in the outputs directory
    print("\nEllipticity arrays saved to outputs/.")
    return results
 
 
if __name__ == "__main__":
    main()

Loaded 100000 galaxy stamps from data/galaxy_stamps.fits

Median HSM galaxy size: sigma_g = 3.436 px
Calibrating weighted-moment response on 12 galaxy sizes...
  hlr=0.40px  sigma_g=0.718px  R=0.2827
  hlr=0.50px  sigma_g=0.732px  R=0.4286
  hlr=0.75px  sigma_g=0.789px  R=0.5862
  hlr=1.00px  sigma_g=0.879px  R=0.5946
  hlr=1.50px  sigma_g=1.139px  R=0.5223
  hlr=2.00px  sigma_g=1.452px  R=0.4475
  hlr=2.50px  sigma_g=1.783px  R=0.3869
  hlr=3.00px  sigma_g=2.119px  R=0.3397
  hlr=4.00px  sigma_g=2.799px  R=0.2727
  hlr=5.00px  sigma_g=3.485px  R=0.2267
  hlr=7.00px  sigma_g=4.862px  R=0.1692
  hlr=10.00px  sigma_g=6.926px  R=0.1224

Calibration range: sigma_g in [0.72, 6.93] px
Calibration R range: [0.122, 0.595]

Summary statistics:
  unweighted  :  92351/100000 valid | <e1> = -0.0002  σ(e1) = 0.1047 | <e2> = +0.0001  σ(e2) = 0.1353
  weighted    :  99997/100000 valid | <e1> = +0.0001  σ(e1) = 0.0503 | <e2> = +0.0000  σ(e2) = 0.0503
  hsm         : 100000/100000 valid | <e1> = +0.000